In [ ]:
# Import necessary libraries

%pip install python-dotenv langchain-openai langchain-community

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 1.8 MB/s eta 0:00:02
   -------------------- ------------------- 1.3/2.5 MB 2.7 MB/s eta 0:00:01
   --------------------------------- ------ 2.1/2.5 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 3.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 10.4 MB/s eta 0:00:02
   -------- ------------------------------- 2.6/12.3 MB 7.4 MB/s eta 0:00:02
   ----------- -------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini",api_key=openai_api_key)

In [ ]:
response = llm.invoke("Suggest me a title of movie in hindi to watch")
response

AIMessage(content='You might enjoy "Chhichhore" – it\'s a heartfelt film that explores friendship, perseverance, and the ups and downs of college life.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-Ct8T6qhj0dXQStTC9pI7zSO0W6y0q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b78b4-19ef-7fa2-872f-c00f7e0a5f57-0', usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]
response = llm.invoke(messages)

In [ ]:
response.content

'Ciao!'

In [ ]:
# Prompt Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [ ]:
prompt = prompt_template.invoke({"language": "hindi", "text": "what is your age"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English into hindi', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is your age', additional_kwargs={}, response_metadata={})])

In [ ]:
llm.invoke(prompt).content

'आपकी उम्र क्या है?'

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("Tell me a joke about {topic}")

prompt = prompt_template.invoke({"topic": "cats"})

response = llm.invoke(prompt)

response.content

'Why did the cat sit on the computer?\n\nBecause it wanted to keep an eye on the mouse!'

In [ ]:
movie_title_template = PromptTemplate.from_template("Suggest me title of a movie to watch in language {language} and of genre {genre}")

movie_title_prompt = movie_title_template.invoke({"language": "hindi", "genre":"comedy"})

response = llm.invoke(movie_title_prompt)

response.content

'You might enjoy watching **"Queen"** (2014). It\'s a hilarious comedy-drama about a young woman who embarks on a solo honeymoon trip to Europe after her marriage falls apart. The film offers a great mix of humor and heartwarming moments.'

In [ ]:
# Pipe Operator Helps us connect flows together . Output of left passed to right - Sequential Chaining 

In [ ]:
from langchain_core.output_parsers import StrOutputParser

movie_title_chain = movie_title_template | llm | StrOutputParser()

movie_title_chain.invoke({"language": "english", "genre":"horror"})

'Sure! You might enjoy **"Hereditary"**. It\'s a psychological horror film that delves into family secrets and trauma, with a deeply unsettling atmosphere. Enjoy your movie night!'

In [ ]:
movie_summary_prompt = PromptTemplate.from_template("Give me 2-3 line summary of the movie {movie_title}")

In [ ]:

composed_chain = {"movie_title": movie_title_chain} | print_title_step | movie_summary_prompt | llm | StrOutputParser()

In [ ]:
composed_chain.invoke({"language": "english", "genre":"horror"})

You might enjoy "The Conjuring" (2013). It's a popular horror film that follows paranormal investigators Ed and Lorraine Warren as they help a family deal with a dark presence in their farmhouse. It's known for its suspenseful atmosphere and chilling moments. Enjoy!


'"None" is a thought-provoking horror film that explores themes of faith, doubt, and the supernatural. It follows a group of individuals who confront their deepest fears and beliefs when they encounter a powerful, malevolent force. As they unravel the mystery behind the entity, they must confront their own inner demons and the implications of their choices.'

In [ ]:
# Runnable Lambda

In [ ]:
from langchain_core.runnables import RunnableLambda

print_title_step = RunnableLambda(lambda x : print(x["movie_title"]))

composed_chain = {"movie_title": movie_title_chain} | print_title_step | movie_summary_prompt | llm | StrOutputParser()

In [ ]:
# Runnable Sequence

In [ ]:
from langchain_core.runnables import RunnableSequence

composed_chain = RunnableSequence({"movie_title": movie_title_chain}, print_title_step, movie_summary_prompt, llm, StrOutputParser())

In [ ]:
summary = composed_chain.invoke({"language": "english", "genre":"horror"})
print(summary)

Sure! I recommend watching **"Hereditary."** It's a psychological horror film that explores themes of family trauma and the dark secrets that can haunt generations. The film has received critical acclaim for its unsettling atmosphere and powerful performances. Enjoy!
"None" follows the story of a troubled priest who wrestles with his faith and purpose after a series of mysterious events challenge his beliefs. As he delves deeper into a world of supernatural occurrences, he must confront both external demons and his own inner turmoil. The film explores themes of redemption, faith, and the struggle between good and evil.


In [ ]:
from langchain_core.runnables import RunnableParallel

translate_hindi_chain = ChatPromptTemplate.from_template("Translate the summary {summary} to hindi") | llm | StrOutputParser()
tranlate_spanish_chain = ChatPromptTemplate.from_template("Translate the summary {summary} to spanish") | llm | StrOutputParser()

translate_runnable = RunnableParallel(hindi_translate = translate_hindi_chain, spanish_translate = tranlate_spanish_chain)

translated_summary = translate_runnable.invoke({"summary" : summary})

print("Hindi Summary: ",  translated_summary["hindi_translate"])
print("Spanish Summary: ", translated_summary["spanish_translate"])

Hindi Summary:  "None" की कहानी एक परेशान पादरी के इर्द-गिर्द घूमती है, जो रहस्यमय घटनाओं की एक श्रृंखला के बाद अपने विश्वास और उद्देश्य के साथ संघर्ष करता है। जब वह अलौकिक घटनाओं की दुनिया में गहराई से उतरता है, तो उसे बाहरी दानवों और अपने भीतर की उथल-पुथल का सामना करना पड़ता है। यह फिल्म त्याग, विश्वास और भले और बुरे के बीच संघर्ष जैसे विषयों की खोज करती है।
Spanish Summary:  "None" sigue la historia de un sacerdote atormentado que lucha con su fe y propósito después de una serie de eventos misteriosos que desafían sus creencias. A medida que se adentra más en un mundo de ocurrencias sobrenaturales, debe enfrentarse tanto a demonios externos como a su propio tumulto interno. La película explora temas de redención, fe y la lucha entre el bien y el mal.
